# PrismML Bonsai 2 27B — FAST Colab benchmark

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vtavakkoli/TinyCeNN-LM/blob/main/notebooks/PrismML_Ternary_Bonsai2_27B_FastEval50_Colab.ipynb)

This version is optimized for **fast setup and fast execution**.

### Fast path
- **No CUDA/C++ compilation**
- Downloads PrismML's **prebuilt CUDA llama.cpp binaries**
- Downloads the smaller **PTQ1_0 (~5.95 GB)** model
- Uses Hugging Face's lightweight **datasets REST API** instead of installing the heavy `datasets` package
- Runs 50 evaluation questions with **4 parallel llama-server slots**
- Reports **pp512**, **tg128**, live prompt tok/s, live generation tok/s, latency, and FastEval-50 accuracy

Model: **`prism-ml/Ternary-Bonsai-2-27B-gguf`**

The only large unavoidable download is the ~5.95 GB model itself.


In [1]:
#@title 1. GPU + fast configuration
import os, sys, json, time, pathlib, subprocess, platform

MODEL_REPO = "prism-ml/Ternary-Bonsai-2-27B-gguf"
MODEL_FILE = "Ternary-Bonsai-2-27B-PTQ1_0.gguf"
GPU_LAYERS = 99
PARALLEL = 4
CTX_TOTAL = 8192   # 4 slots => ~2048 tokens/slot
SERVER_PORT = None  # auto-selected free localhost port in server cell
SEED = 20260919
N_PER_BENCH = 10

RESULT_DIR = pathlib.Path("/content/prism_bonsai2_fasteval50")
RESULT_DIR.mkdir(parents=True, exist_ok=True)

if subprocess.run(["bash","-lc","command -v nvidia-smi"], capture_output=True).returncode != 0:
    raise RuntimeError("No NVIDIA GPU found. In Colab choose Runtime → Change runtime type → GPU.")

subprocess.run(["nvidia-smi"], check=True)
print("Python:", sys.version.split()[0])
print("Model :", MODEL_REPO)
print("GGUF  :", MODEL_FILE)
print("Tests :", N_PER_BENCH * 5)


Python: 3.13.15
Model : prism-ml/Ternary-Bonsai-2-27B-gguf
GGUF  : Ternary-Bonsai-2-27B-PTQ1_0.gguf
Tests : 50


In [2]:
#@title 2. FAST install — prebuilt PrismML CUDA binaries, NO compilation
import os, sys, subprocess, pathlib

# Minimal Python packages only.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-U",
     "huggingface_hub", "hf_xet", "requests", "pandas"],
    check=True,
)
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"

DEMO_DIR = pathlib.Path("/content/Bonsai-demo")
if DEMO_DIR.exists():
    subprocess.run(["git","-C",str(DEMO_DIR),"fetch","--depth","1","origin"], check=True)
    subprocess.run(["git","-C",str(DEMO_DIR),"reset","--hard","origin/main"], check=True)
else:
    subprocess.run(
        ["git","clone","--depth","1","https://github.com/PrismML-Eng/Bonsai-demo.git",str(DEMO_DIR)],
        check=True,
    )

# PrismML's script detects the Colab CUDA version and downloads the matching
# prebuilt llama.cpp archive. This is much faster than cmake + CUDA compilation.
subprocess.run(
    ["sh", str(DEMO_DIR/"scripts"/"download_binaries.sh")],
    cwd=str(DEMO_DIR),
    check=True,
)

candidate_dirs = [
    DEMO_DIR/"bin"/"cuda",
    DEMO_DIR/"bin"/"vulkan",
    DEMO_DIR/"bin"/"cpu",
]
BIN_DIR = next((p for p in candidate_dirs if (p/"llama-server").exists()), None)
if BIN_DIR is None:
    raise RuntimeError("PrismML prebuilt llama-server was not found.")

LLAMA_SERVER = BIN_DIR/"llama-server"
LLAMA_BENCH = BIN_DIR/"llama-bench"
assert LLAMA_SERVER.exists()
assert LLAMA_BENCH.exists()

os.environ["LD_LIBRARY_PATH"] = str(BIN_DIR) + (
    ":" + os.environ["LD_LIBRARY_PATH"] if os.environ.get("LD_LIBRARY_PATH") else ""
)

print("✓ Fast runtime ready")
print("Binary dir:", BIN_DIR)
subprocess.run([str(LLAMA_SERVER), "--version"], check=False)


✓ Fast runtime ready
Binary dir: /content/Bonsai-demo/bin/cuda


CompletedProcess(args=['/content/Bonsai-demo/bin/cuda/llama-server', '--version'], returncode=0)

In [3]:
#@title 3. Download the 5.95 GB PTQ1_0 model
from huggingface_hub import hf_hub_download
import pathlib, os, time

MODEL_DIR = pathlib.Path("/content/models/prism-bonsai2-27b")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

t0 = time.perf_counter()
MODEL_PATH = pathlib.Path(hf_hub_download(
    repo_id=MODEL_REPO,
    filename=MODEL_FILE,
    local_dir=str(MODEL_DIR),
))
download_s = time.perf_counter() - t0

print("✓ Model ready:", MODEL_PATH)
print(f"Size: {MODEL_PATH.stat().st_size / 1024**3:.2f} GiB")
print(f"Download/check time: {download_s:.1f} s")


Ternary-Bonsai-2-27B-PTQ1_0.gguf: reconstructing file:   0%|          |  0.00B / 5.95GB            

Ternary-Bonsai-2-27B-PTQ1_0.gguf: downloading bytes:           |  0.00B            

✓ Model ready: /content/models/prism-bonsai2-27b/Ternary-Bonsai-2-27B-PTQ1_0.gguf
Size: 5.54 GiB
Download/check time: 50.0 s


In [4]:
#@title 4. Pure throughput — llama-bench pp512 + tg128
import subprocess, json, pandas as pd
from IPython.display import display

cmd = [
    str(LLAMA_BENCH),
    "-m", str(MODEL_PATH),
    "-ngl", str(GPU_LAYERS),
    "-r", "3",
    "-o", "json",
]
print("Running:", " ".join(cmd))
p = subprocess.run(cmd, capture_output=True, text=True, env=os.environ.copy())

(RESULT_DIR/"llama_bench_stdout.txt").write_text(p.stdout)
(RESULT_DIR/"llama_bench_stderr.txt").write_text(p.stderr)

if p.returncode != 0:
    print(p.stdout)
    print(p.stderr[-5000:])
    raise RuntimeError(f"llama-bench failed: {p.returncode}")

raw = p.stdout.strip()
try:
    bench_json = json.loads(raw)
except json.JSONDecodeError:
    starts = [x for x in (raw.find("["), raw.find("{")) if x >= 0]
    start = min(starts)
    end = max(raw.rfind("]"), raw.rfind("}"))
    bench_json = json.loads(raw[start:end+1])

rows = bench_json if isinstance(bench_json, list) else [bench_json]
bench_df = pd.DataFrame(rows)
display(bench_df)

def find_metric(kind):
    for r in rows:
        nprompt = int(r.get("n_prompt", 0) or 0)
        ngen = int(r.get("n_gen", 0) or 0)
        if kind == "pp512" and nprompt == 512 and ngen == 0:
            return float(r["avg_ts"])
        if kind == "tg128" and ngen == 128:
            return float(r["avg_ts"])
    return None

PP512 = find_metric("pp512")
TG128 = find_metric("tg128")
print(f"\npp512: {PP512 if PP512 is not None else 'see table'} tok/s")
print(f"tg128 : {TG128 if TG128 is not None else 'see table'} tok/s")

bench_df.to_csv(RESULT_DIR/"llama_bench.csv", index=False)


Running: /content/Bonsai-demo/bin/cuda/llama-bench -m /content/models/prism-bonsai2-27b/Ternary-Bonsai-2-27B-PTQ1_0.gguf -ngl 99 -r 3 -o json


,build_commit,build_number,cpu_info,gpu_info,backends,model_filename,model_type,model_size,model_n_params,n_batch,...,n_prompt,n_gen,n_depth,test_time,avg_ns,stddev_ns,avg_ts,stddev_ts,samples_ns,samples_ts
0,9a9394a89,10709,Intel(R) Xeon(R) CPU @ 2.00GHz,Tesla T4,CUDA,/content/models/prism-bonsai2-27b/Ternary-Bons...,qwen35 27B PTQ1_0 - 1.75 bpw ternary (group 128),5935527936,26895998464,2048,...,512,0,0,2026-09-19T17:00:53Z,2611232980,40134125,196.106613,2.992088,"[2656697886, 2596275629, 2580725426]","[192.72, 197.206, 198.394]"
1,9a9394a89,10709,Intel(R) Xeon(R) CPU @ 2.00GHz,Tesla T4,CUDA,/content/models/prism-bonsai2-27b/Ternary-Bons...,qwen35 27B PTQ1_0 - 1.75 bpw ternary (group 128),5935527936,26895998464,2048,...,0,128,0,2026-09-19T17:01:53Z,8424690366,103101046,15.194950,0.185442,"[8330716171, 8408380897, 8534974031]","[15.3648, 15.2229, 14.9971]"



pp512: 196.106613 tok/s
tg128 : 15.19495 tok/s


In [5]:
#@title 5. Start fast local server — 4 parallel slots, thinking OFF
import subprocess, time, requests, os, socket

SERVER_LOG = RESULT_DIR/"llama_server.log"

# Stop only a server started by an earlier execution of this notebook cell.
if "server_proc" in globals() and server_proc.poll() is None:
    server_proc.terminate()
    try:
        server_proc.wait(timeout=8)
    except subprocess.TimeoutExpired:
        server_proc.kill()

# Colab/Jupyter or another process may already own 8080.
# Ask the OS for a currently free localhost port instead of hard-coding one.
def get_free_local_port():
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.bind(("127.0.0.1", 0))
        return int(s.getsockname()[1])

SERVER_PORT = get_free_local_port()
SERVER_URL = f"http://127.0.0.1:{SERVER_PORT}"

log_handle = open(SERVER_LOG, "w")
server_cmd = [
    str(LLAMA_SERVER),
    "-m", str(MODEL_PATH),
    "--host", "127.0.0.1",
    "--port", str(SERVER_PORT),
    "-ngl", str(GPU_LAYERS),
    "-fa", "on",
    "-c", str(CTX_TOTAL),
    "-np", str(PARALLEL),
    "--jinja",
    "--reasoning", "off",
    "--temp", "0.7",
    "--top-p", "0.80",
    "--top-k", "20",
]

print(f"Starting PrismML server on free port {SERVER_PORT}...")
server_proc = subprocess.Popen(
    server_cmd,
    stdout=log_handle,
    stderr=subprocess.STDOUT,
    env=os.environ.copy(),
)

ready = False
for _ in range(180):
    if server_proc.poll() is not None:
        break
    try:
        if requests.get(SERVER_URL + "/health", timeout=1).ok:
            ready = True
            break
    except requests.RequestException:
        pass
    time.sleep(1)

if not ready:
    log_handle.flush()
    print(SERVER_LOG.read_text()[-8000:])
    raise RuntimeError(f"Server did not become ready on port {SERVER_PORT}.")

models = requests.get(SERVER_URL + "/v1/models", timeout=20).json()
served_model = models["data"][0]["id"]
print("✓ Server ready:", SERVER_URL)
print("Model:", served_model)
print("Parallel slots:", PARALLEL)


Starting PrismML server on free port 43779...
✓ Server ready: http://127.0.0.1:43779
Model: /content/models/prism-bonsai2-27b/Ternary-Bonsai-2-27B-PTQ1_0.gguf
Parallel slots: 4


In [6]:
#@title 6. Live token/s measurement
import requests, json, pandas as pd
from IPython.display import display

payload = {
    "prompt": (
        "Explain in concise technical terms why ternary low-bit weights can reduce "
        "memory bandwidth pressure during autoregressive inference. Include one limitation."
    ),
    "n_predict": 128,
    "temperature": 0.0,
    "cache_prompt": False,
}
r = requests.post(SERVER_URL + "/completion", json=payload, timeout=600)
r.raise_for_status()
d = r.json()
timings = d.get("timings", {})

live_speed = {
    "prompt_tokens": timings.get("prompt_n"),
    "prompt_tok_s": timings.get("prompt_per_second"),
    "generated_tokens": timings.get("predicted_n"),
    "generation_tok_s": timings.get("predicted_per_second"),
    "prompt_ms": timings.get("prompt_ms"),
    "generation_ms": timings.get("predicted_ms"),
}
display(pd.DataFrame([live_speed]))

print("Prompt tok/s    :", live_speed["prompt_tok_s"])
print("Generation tok/s:", live_speed["generation_tok_s"])
print("\nOutput preview:\n", d.get("content","")[:800])

with open(RESULT_DIR/"live_speed.json","w") as f:
    json.dump({"metrics": live_speed, "raw_timings": timings}, f, indent=2)


,prompt_tokens,prompt_tok_s,generated_tokens,generation_tok_s,prompt_ms,generation_ms
0,26,45.565272,128,14.762929,570.61,8602.629


Prompt tok/s    : 45.56527225250171
Generation tok/s: 14.762928867442731

Output preview:
 

<think>

</think>

Ternary low-bit weights (e.g., {-1, 0, +1}) reduce memory bandwidth pressure during autoregressive inference by significantly decreasing the storage footprint per parameter (from 32-bit FP32 to 1-bit or 2-bit representations). This compression allows more model parameters to fit within the limited capacity of high-bandwidth memory (HBM) or on-chip SRAM, reducing the frequency of data transfers between slower off-chip memory and the compute units. Additionally, the sparsity inherent in ternary weights (often ~50% zeros) enables the use of sparse matrix


In [7]:
#@title 7. Fetch 50 benchmark questions — lightweight REST API, no datasets install
import requests, random, subprocess, sys

ROWS_API = "https://datasets-server.huggingface.co/rows"

def fetch_pool(dataset, config, split, n=100):
    params = {
        "dataset": dataset,
        "config": config,
        "split": split,
        "offset": 0,
        "length": n,
    }
    r = requests.get(ROWS_API, params=params, timeout=120)
    r.raise_for_status()
    return [x["row"] for x in r.json()["rows"]]

def choose(rows, n, seed):
    return random.Random(seed).sample(rows, min(n, len(rows)))

def mc_prompt(stem, labels, options):
    choices = "\n".join(f"{lab}. {opt}" for lab, opt in zip(labels, options))
    return f"{stem.strip()}\n\n{choices}\n\nAnswer with only one label: {', '.join(labels)}."

eval_items = []

# ARC-Challenge
rows = choose(fetch_pool("allenai/ai2_arc", "ARC-Challenge", "validation"), N_PER_BENCH, SEED+1)
for x in rows:
    labels = [str(v) for v in x["choices"]["label"]]
    opts = [str(v) for v in x["choices"]["text"]]
    eval_items.append({"benchmark":"ARC-Challenge","prompt":mc_prompt(x["question"],labels,opts),"labels":labels,"gold":str(x["answerKey"])})

# BoolQ
rows = choose(fetch_pool("google/boolq", "default", "validation"), N_PER_BENCH, SEED+2)
for x in rows:
    labels = ["A","B"]
    stem = f'Passage: {x["passage"]}\n\nQuestion: {x["question"]}'
    eval_items.append({"benchmark":"BoolQ","prompt":mc_prompt(stem,labels,["Yes","No"]),"labels":labels,"gold":"A" if x["answer"] else "B"})

# OpenBookQA
rows = choose(fetch_pool("allenai/openbookqa", "main", "validation"), N_PER_BENCH, SEED+3)
for x in rows:
    labels = [str(v) for v in x["choices"]["label"]]
    opts = [str(v) for v in x["choices"]["text"]]
    eval_items.append({"benchmark":"OpenBookQA","prompt":mc_prompt(x["question_stem"],labels,opts),"labels":labels,"gold":str(x["answerKey"])})

# HellaSwag
rows = choose(fetch_pool("Rowan/hellaswag", "default", "validation"), N_PER_BENCH, SEED+4)
for x in rows:
    labels = ["A","B","C","D"]
    stem = (str(x.get("ctx_a","")) + " " + str(x.get("ctx_b",""))).strip()
    opts = [str(v) for v in x["endings"]]
    gold = labels[int(x["label"])]
    eval_items.append({"benchmark":"HellaSwag","prompt":mc_prompt(stem,labels,opts),"labels":labels,"gold":gold})

# WinoGrande
rows = choose(fetch_pool("allenai/winogrande", "winogrande_xl", "validation"), N_PER_BENCH, SEED+5)
for x in rows:
    labels = ["A","B"]
    stem = str(x["sentence"]).replace("_","_____")
    gold = "A" if str(x["answer"]) == "1" else "B"
    eval_items.append({"benchmark":"WinoGrande","prompt":mc_prompt(stem,labels,[x["option1"],x["option2"]]),"labels":labels,"gold":gold})

assert len(eval_items) == 50, len(eval_items)
print("✓ FastEval-50 ready")
from collections import Counter
print(Counter(x["benchmark"] for x in eval_items))


✓ FastEval-50 ready
Counter({'ARC-Challenge': 10, 'BoolQ': 10, 'OpenBookQA': 10, 'HellaSwag': 10, 'WinoGrande': 10})


In [8]:
#@title 8. FastEval-50 — 4 requests in parallel
import requests, time, re, pandas as pd, json, numpy as np
from concurrent.futures import ThreadPoolExecutor, as_completed
from IPython.display import display

SYSTEM = "Return only the requested multiple-choice option label. No explanation."

def parse_label(text, allowed):
    text = (text or "").strip().upper()
    compact = re.sub(r"[^A-Z0-9]", "", text)
    if compact in allowed:
        return compact
    for lab in allowed:
        if re.search(rf"(?<![A-Z0-9]){re.escape(lab)}(?![A-Z0-9])", text):
            return lab
    return ""

def ask_one(idx_item):
    idx, item = idx_item
    payload = {
        "model": served_model,
        "messages": [
            {"role":"system","content":SYSTEM},
            {"role":"user","content":item["prompt"]},
        ],
        "temperature": 0.0,
        "max_tokens": 8,
    }
    t0 = time.perf_counter()
    r = requests.post(SERVER_URL + "/v1/chat/completions", json=payload, timeout=600)
    latency = time.perf_counter() - t0
    r.raise_for_status()
    data = r.json()
    text = data["choices"][0]["message"].get("content") or ""
    pred = parse_label(text, item["labels"])
    usage = data.get("usage") or {}
    return {
        "id": idx + 1,
        "benchmark": item["benchmark"],
        "gold": item["gold"],
        "prediction": pred,
        "correct": pred == item["gold"],
        "raw_output": text,
        "latency_s": latency,
        "prompt_tokens": int(usage.get("prompt_tokens") or 0),
        "completion_tokens": int(usage.get("completion_tokens") or 0),
    }

t_all = time.perf_counter()
records = []

with ThreadPoolExecutor(max_workers=PARALLEL) as ex:
    futures = [ex.submit(ask_one, x) for x in enumerate(eval_items)]
    done = 0
    for fut in as_completed(futures):
        row = fut.result()
        records.append(row)
        done += 1
        mark = "✓" if row["correct"] else "✗"
        print(f'{done:02d}/50  {row["benchmark"]:<14} gold={row["gold"]:<2} pred={row["prediction"] or "?":<2} {mark}')

wall_s = time.perf_counter() - t_all
results_df = pd.DataFrame(records).sort_values("id").reset_index(drop=True)

summary_df = results_df.groupby("benchmark").agg(
    questions=("correct","size"),
    correct=("correct","sum"),
    accuracy=("correct","mean"),
    avg_latency_s=("latency_s","mean"),
)
summary_df.loc["OVERALL"] = {
    "questions": len(results_df),
    "correct": int(results_df["correct"].sum()),
    "accuracy": float(results_df["correct"].mean()),
    "avg_latency_s": float(results_df["latency_s"].mean()),
}

display(summary_df.style.format({"accuracy":"{:.1%}","avg_latency_s":"{:.2f}"}))

overall_acc = float(results_df["correct"].mean())
print(f"\nOVERALL: {int(results_df['correct'].sum())}/50 = {overall_acc:.1%}")
print(f"FastEval wall time: {wall_s:.1f}s")
print(f"Throughput: {50/wall_s:.3f} questions/s")
print(f"Mean request latency: {results_df['latency_s'].mean():.2f}s")

results_df.to_csv(RESULT_DIR/"fasteval50_results.csv", index=False)
summary_df.to_csv(RESULT_DIR/"fasteval50_summary.csv")

report = {
    "model": MODEL_REPO,
    "file": MODEL_FILE,
    "gpu_layers": GPU_LAYERS,
    "parallel_slots": PARALLEL,
    "pp512_tok_s": PP512,
    "tg128_tok_s": TG128,
    "live_prompt_tok_s": live_speed.get("prompt_tok_s"),
    "live_generation_tok_s": live_speed.get("generation_tok_s"),
    "fasteval_correct": int(results_df["correct"].sum()),
    "fasteval_total": 50,
    "fasteval_accuracy": overall_acc,
    "fasteval_wall_s": wall_s,
}
with open(RESULT_DIR/"report.json","w") as f:
    json.dump(report, f, indent=2)

print("\nFINAL")
for k,v in report.items():
    print(f"{k}: {v}")


01/50  ARC-Challenge  gold=A  pred=A  ✓
02/50  ARC-Challenge  gold=A  pred=A  ✓
03/50  ARC-Challenge  gold=B  pred=B  ✓
04/50  ARC-Challenge  gold=C  pred=C  ✓
05/50  ARC-Challenge  gold=D  pred=D  ✓
06/50  ARC-Challenge  gold=B  pred=B  ✓
07/50  ARC-Challenge  gold=C  pred=C  ✓
08/50  ARC-Challenge  gold=D  pred=A  ✗
09/50  ARC-Challenge  gold=B  pred=B  ✓
10/50  BoolQ          gold=A  pred=A  ✓
11/50  ARC-Challenge  gold=B  pred=B  ✓
12/50  BoolQ          gold=A  pred=A  ✓
13/50  BoolQ          gold=A  pred=B  ✗
14/50  BoolQ          gold=A  pred=A  ✓
15/50  BoolQ          gold=A  pred=A  ✓
16/50  BoolQ          gold=B  pred=B  ✓
17/50  BoolQ          gold=B  pred=B  ✓
18/50  BoolQ          gold=A  pred=A  ✓
19/50  BoolQ          gold=A  pred=A  ✓
20/50  BoolQ          gold=A  pred=A  ✓
21/50  OpenBookQA     gold=C  pred=C  ✓
22/50  OpenBookQA     gold=B  pred=C  ✗
23/50  OpenBookQA     gold=B  pred=B  ✓
24/50  OpenBookQA     gold=B  pred=B  ✓
25/50  OpenBookQA     gold=D  pred=D  ✓


,questions,correct,accuracy,avg_latency_s
benchmark,,,,
ARC-Challenge,10,9,90.0%,5.79
BoolQ,10,9,90.0%,8.16
HellaSwag,10,8,80.0%,6.44
OpenBookQA,10,9,90.0%,5.68
WinoGrande,10,9,90.0%,5.12
OVERALL,50,44,88.0%,6.24



OVERALL: 44/50 = 88.0%
FastEval wall time: 80.0s
Throughput: 0.625 questions/s
Mean request latency: 6.24s

FINAL
model: prism-ml/Ternary-Bonsai-2-27B-gguf
file: Ternary-Bonsai-2-27B-PTQ1_0.gguf
gpu_layers: 99
parallel_slots: 4
pp512_tok_s: 196.106613
tg128_tok_s: 15.19495
live_prompt_tok_s: 45.56527225250171
live_generation_tok_s: 14.762928867442731
fasteval_correct: 44
fasteval_total: 50
fasteval_accuracy: 0.88
fasteval_wall_s: 79.98409380400005


In [19]:
#@title 9. Coding test — Snake HTML, 128K context
import requests, time, socket, subprocess, os, html as html_lib
from IPython.display import display, HTML

# Stop previous server
if "server_proc" in globals() and server_proc.poll() is None:
    server_proc.terminate()
    try:
        server_proc.wait(timeout=8)
    except subprocess.TimeoutExpired:
        server_proc.kill()

# Find free port
def free_port():
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.bind(("127.0.0.1", 0))
        return s.getsockname()[1]

SERVER_PORT = free_port()
SERVER_URL = f"http://127.0.0.1:{SERVER_PORT}"
CODE_LOG = RESULT_DIR / "snake_code_server.log"

log_handle = open(CODE_LOG, "w")

# 128K context, single slot
server_cmd = [
    str(LLAMA_SERVER),
    "-m", str(MODEL_PATH),
    "--host", "127.0.0.1",
    "--port", str(SERVER_PORT),
    "-ngl", str(GPU_LAYERS),
    "-fa", "on",
    "-c", "131072",
    "-np", "1",
    "--jinja",
    "--reasoning", "off",
]

print(f"Starting coding server: 1 slot, 128K context, port {SERVER_PORT}...")

server_proc = subprocess.Popen(
    server_cmd,
    stdout=log_handle,
    stderr=subprocess.STDOUT,
    env=os.environ.copy(),
)

# Wait for server
for _ in range(180):
    if server_proc.poll() is not None:
        break
    try:
        if requests.get(SERVER_URL + "/health", timeout=1).ok:
            break
    except requests.RequestException:
        pass
    time.sleep(1)
else:
    log_handle.flush()
    print(CODE_LOG.read_text()[-8000:])
    raise RuntimeError("Server did not start")

models = requests.get(
    SERVER_URL + "/v1/models",
    timeout=20
).json()

served_model = models["data"][0]["id"]

print("✓ Model:", served_model)
print("✓ Context: 131072")

SYSTEM = """
You are an expert JavaScript and HTML game programmer.
Return only complete executable HTML.
"""

PROMPT = """
Create a complete playable Snake game as ONE standalone HTML file.

Return ONLY the HTML source.
Do not use Markdown fences.
Do not explain anything.

Use:
- HTML
- CSS
- JavaScript

Everything must be inside one HTML file.

The game must include:
- playable Snake
- keyboard controls
- WASD controls
- mobile touch controls
- score
- high score
- restart button
- game-over screen
- responsive canvas

Make sure the final game actually works in a browser.
"""

t0 = time.perf_counter()

response = requests.post(
    SERVER_URL + "/v1/chat/completions",
    json={
        "model": served_model,
        "messages": [
            {
                "role": "system",
                "content": SYSTEM
            },
            {
                "role": "user",
                "content": PROMPT
            }
        ],
        "temperature": 0.3,
        "top_p": 0.9,
        "top_k": 20,
        "max_tokens": 65536,
    },
    timeout=1800,
)

response.raise_for_status()

data = response.json()

snake_html = data["choices"][0]["message"]["content"]

elapsed = time.perf_counter() - t0

usage = data.get("usage") or {}
tokens = int(usage.get("completion_tokens") or 0)

# Save exactly what model returned
SNAKE_FILE = RESULT_DIR / "snake_game.html"
SNAKE_FILE.write_text(
    snake_html,
    encoding="utf-8"
)

print("\n✓ Generation finished")
print("Generated tokens:", tokens)
print("Time:", f"{elapsed:.2f} sec")

if elapsed > 0:
    print("Tokens/s:", f"{tokens / elapsed:.2f}")

print("Saved:", SNAKE_FILE)

# Preview
display(
    HTML(
        f'<iframe '
        f'srcdoc="{html_lib.escape(snake_html, quote=True)}" '
        f'style="width:100%;height:700px;border:1px solid #aaa;">'
        f'</iframe>'
    )
)

# Download
from google.colab import files
files.download(str(SNAKE_FILE))

Starting coding server: 1 slot, 128K context, port 38691...
✓ Model: /content/models/prism-bonsai2-27b/Ternary-Bonsai-2-27B-PTQ1_0.gguf
✓ Context: 131072

✓ Generation finished
Generated tokens: 3764
Time: 280.66 sec
Tokens/s: 13.41
Saved: /content/prism_bonsai2_fasteval50/snake_game.html


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>